# **Random Meal Planner**

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from datetime import datetime, timedelta
from random import randint, random, seed

from engines import RandomMealPlanner, get_pantry_ingredient, load_all_ingredients, make_preferences
from models import (
    MealPlanningEnvironment,
    Pantry,
)

In [2]:
seed(1)

In [3]:
preferences = make_preferences()

In [4]:
all_ingredients = load_all_ingredients("../recipe_extraction/supplemented_structured_ingredients.json")

In [5]:
CURRENT_DATE = datetime.now()

In [6]:
pantry_ingredient_name_by_quantity = {
    "chicken breast": 800,
    "broccoli": 1500,
    "rice": 1000,
}

In [7]:
pantry_ingredients = [
    get_pantry_ingredient(name, CURRENT_DATE + timedelta(days=randint(1, 7)), all_ingredients)
    for name in pantry_ingredient_name_by_quantity.keys()
]

In [8]:
pantry_ingredient_by_quantity = dict(zip(pantry_ingredients, pantry_ingredient_name_by_quantity.values()))

In [9]:
pantry = Pantry()

for pantry_ingredient, quantity in pantry_ingredient_by_quantity.items():
    pantry.add(
        pantry_ingredient,
        quantity,
    )

In [22]:
pantry.print()

---
Quantity: 800 g
Ingredient: chicken breast
	Estimated Expiration Date: 2026-04-30
	Nutritional Information:
		Calories: 125.0 kcal
		Carbohydrates: 1.79 g
		Sugar: 0.0 g
		Protein: 16.07 g
		Fat: 5.36 g
		Saturated Fat: 1.79 g
		Fiber: 1.8 g
		Sodium: 571.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: No
		Vegan: No
---
---
Quantity: 1500 g
Ingredient: broccoli
	Estimated Expiration Date: 2026-05-03
	Nutritional Information:
		Calories: 31.0 kcal
		Carbohydrates: 6.27 g
		Sugar: 1.4 g
		Protein: 2.57 g
		Fat: 0.34 g
		Saturated Fat: 0.039 g
		Fiber: 2.4 g
		Sodium: 36.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 1000 g
Ingredient: rice
	Estimated Expiration Date: 2026-05-05
	Nutritional Information:
		Calories: 356.0 kcal
		Carbohydrates: 80.0 g
		Sugar: 0.0 g
		Protein: 6.67 g
		Fat: 0.0 g
		Saturated Fat: 0.0 g
		Fiber: 2.2 g
		Sodium: 0.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---


In [11]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=[],
    pantry=pantry,
    preferences=preferences,
)

In [12]:
meal_planning_environment.load_recipes_from_json("../recipe_extraction/supplemented_structured_recipes.json")

In [13]:
all_ingredient_names = []

for recipe in meal_planning_environment.recipes:
    for ingredient_name in recipe.ingredients.keys():
        all_ingredient_names.append(ingredient_name)

In [14]:
all_ingredient_costs = {}

for ingredient_name in sorted(set(all_ingredient_names)):
    all_ingredient_costs[ingredient_name] = random()

In [15]:
meal_planning_environment.ingredient_costs = all_ingredient_costs
meal_planning_environment._check_ingredient_costs()

In [16]:
planner = RandomMealPlanner(meal_planning_environment, random_seed=1)

In [17]:
best_meal_plan = planner.generate_meal_plan()

In [18]:
pantry_consumption_df = planner.get_pantry_consumption()
pantry_consumption_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,800,0.0,800.0,1d
1,broccoli,1500,0.0,1500.0,4d
2,rice,1000,0.0,1000.0,6d


In [19]:
meal_plan_df = planner.get_meal_plan_recipes()
meal_plan_df

,Day,Meal 1,Meal 2,Meal 3
0,1,Baguette Toasts,Tarragon-Roasted Halibut with Hazelnut Brown B...,Raspberry Custard
1,2,Roasted Sesame-and Panko-Coated Asparagus with...,Key Lime Pie with Almond Crumb Crust,Sephardic Spinach Patties
2,3,"Root Vegetable ""Lasagna"" with Mushroom Broth",Macaroni and Blue Cheese with Chives,Broccoli Rabe with Pine Nuts and Raisins
3,4,Jícama-Melon Salad,Tropical Fruit Salad,Garnet Yam Puree
4,5,Braised Chicken with Dates and Moroccan Spices,Fried Anchovies with Salbixtada Sauce,"Zucchini, Potato, and Cilantro Soup"
5,6,Pear-Hazelnut Cheesecakes with Pear-Raspberry ...,Spiced Tuna Steaks with Fennel and Red Peppers,Mussels with Coconut Curry Sauce
6,7,Sea Bass with Moroccan Salsa,A Simple Roast Turkey,Roasted Tomato-Chipotle Salsa


In [20]:
shopping_list_df, num_ingredients, total_cost = planner.get_shopping_list()

print(f"Total ingredients needed to purchase: {num_ingredients}")
print(f"Total cost: €{total_cost:.2f}")

shopping_list_df

Total ingredients needed to purchase: 139
Total cost: €78.21


,Ingredient,Quantity to Buy (g),Cost (€)
0,(generous) fennel seeds,1.2,0.00
1,Asian sesame oil,2.2,0.00
2,Coarse kosher salt,100.0,0.42
3,Fresh chives,3.0,0.03
4,Fresh mint,33.3,0.18
...,...,...,...
135,white onion,4.3,0.04
136,whole milk,47.3,0.36
137,zucchini,453.6,0.58
138,zweiback crumbs,14.8,0.09


In [21]:
daily_nutrition_df = planner.get_daily_nutrition()
daily_nutrition_df

,Calories,Protein,Δ Calories and Target Calories,Δ Protein and Target Protein
Day 1,1621.6 kcal,34.1 g,-378.4 kcal,-15.9 g
Day 2,761.7 kcal,20.6 g,-1238.3 kcal,-29.4 g
Day 3,3520.5 kcal,122.4 g,1520.5 kcal,72.4 g
Day 4,3408.0 kcal,32.9 g,1408.0 kcal,-17.1 g
Day 5,8673.2 kcal,401.2 g,6673.2 kcal,351.2 g
Day 6,1328.9 kcal,49.9 g,-671.1 kcal,-0.1 g
Day 7,743.1 kcal,25.2 g,-1256.9 kcal,-24.8 g
